# ICOtest Report Analysis

This notebook loads and analyzes all JSON test reports from the `reports/` directory.

**Features:**
- Load all test reports into a tabular DataFrame
- Store time series data separately for efficient analysis
- Interactive filtering by device and date range
- Visualize power usage trends and device comparisons
- Access raw time series data for detailed inspection

## 1. Setup - Import Libraries and Define Helpers

In [10]:
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [11]:
# Define data loading classes for device-centric analysis

class TestDataLoader:
    """Load and normalize test reports into device-centric tables"""
    
    def __init__(self, reports_dir='../reports'):
        self.reports_dir = Path(reports_dir)
        self.reports_df = None
        self.tests_df = None
        self.measurements_df = None
        self.series_store = {}
        self.report_files = []
    
    def find_reports(self):
        """Find all JSON report files"""
        hardware_test_files = glob.glob(str(self.reports_dir / 'hardware_test_*.json'))
        icotronic_files = glob.glob(str(self.reports_dir / 'ICOtronic_*.json'))
        renamed_files = glob.glob(str(self.reports_dir / '*_*_*_*.json'))
        
        all_files = sorted(set(hardware_test_files + icotronic_files + renamed_files))
        self.report_files = [f for f in all_files if not f.endswith('.log')]
        return len(self.report_files)
    
    def load_all_reports(self):
        """Load all reports and create normalized tables"""
        reports = []
        tests = []
        measurements = []
        
        for report_file in self.report_files:
            try:
                with open(report_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                filename = Path(report_file).stem
                report_info = self._parse_report(filename, data)
                reports.append(report_info)
                
                for test in data.get('tests', []):
                    test_info, test_measurements, series_data = self._parse_test(
                        test, report_info['report_id'], report_info['device_id'], 
                        report_info['created_dt']
                    )
                    tests.append(test_info)
                    measurements.extend(test_measurements)
                    
                    if series_data:
                        key = (report_info['device_id'], report_info['report_id'], test_info['test_name'])
                        self.series_store[key] = series_data
                        
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        
        self.reports_df = pd.DataFrame(reports)
        self.tests_df = pd.DataFrame(tests)
        self.measurements_df = pd.DataFrame(measurements) if measurements else pd.DataFrame()
        
        return len(reports), len(tests), len(measurements)
    
    def _parse_report(self, filename, data):
        """Extract report-level information"""
        timestamp = data.get('created')
        timestamp_dt = datetime.fromtimestamp(timestamp) if timestamp else None
        
        run_id = filename
        device_id = self._extract_device_id(filename, data)
        
        return {
            'report_id': run_id,
            'device_id': device_id,
            'created': timestamp,
            'created_dt': timestamp_dt,
            'duration': data.get('duration'),
            'exitcode': data.get('exitcode'),
            'total_tests': data.get('summary', {}).get('total', 0),
            'passed_tests': data.get('summary', {}).get('passed', 0),
            'failed_tests': data.get('summary', {}).get('failed', 0),
            'is_legacy': device_id is None or device_id == filename
        }
    
    def _extract_device_id(self, filename, data):
        """Extract device_id from filename or test metadata"""
        parts = filename.split('_')
        
        if len(parts) >= 2:
            prefix = parts[0]
            if prefix in ['hardware', 'ICOtronic']:
                return None
            return prefix
        
        for test in data.get('tests', []):
            meta = test.get('metadata', {})
            if 'sensor_mac_base64' in meta:
                return meta['sensor_mac_base64']
        
        return None
    
    def _parse_test(self, test, report_id, device_id, created_dt):
        """Parse individual test and its measurements"""
        nodeid = test.get('nodeid', '')
        test_name = nodeid.split('::')[-1] if '::' in nodeid else nodeid
        
        duration = None
        if test.get('call') and isinstance(test['call'], dict):
            duration = test['call'].get('duration')
        
        test_info = {
            'report_id': report_id,
            'device_id': device_id,
            'test_name': test_name,
            'nodeid': nodeid,
            'outcome': test.get('outcome', 'unknown'),
            'duration_s': duration,
            'created_dt': created_dt
        }
        
        test_measurements = []
        series_data = {}
        
        metadata = test.get('metadata', {})
        if metadata:
            for key, value in metadata.items():
                if key == 'sensor_mac_base64':
                    test_info['device_id'] = value
                    continue
                
                if isinstance(value, list) and len(value) > 10:
                    series_data[key] = np.array(value)
                    continue
                
                if isinstance(value, dict) and 'value' in value:
                    measurement = {
                        'report_id': report_id,
                        'device_id': device_id,
                        'test_name': test_name,
                        'metric': key,
                        'value': value.get('value'),
                        'unit': value.get('unit', ''),
                        'lower_limit': value.get('lower_limit'),
                        'upper_limit': value.get('upper_limit'),
                        'description': value.get('description', '')
                    }
                    test_measurements.append(measurement)
                
                elif isinstance(value, str):
                    test_info[f'{key}_text'] = value
        
        return test_info, test_measurements, series_data
    
    def get_device_summary(self):
        """Get device-level summary statistics"""
        if self.reports_df is None:
            return None
        
        summary = self.reports_df.groupby('device_id').agg(
            num_runs=('report_id', 'count'),
            total_tests=('total_tests', 'sum'),
            total_passed=('passed_tests', 'sum'),
            total_failed=('failed_tests', 'sum'),
            date_range=('created_dt', lambda x: f"{x.min().date()} to {x.max().date()}")
        ).reset_index()
        
        summary['pass_rate'] = (summary['total_passed'] / summary['total_tests'] * 100).round(1)
        
        return summary
    
    def get_test_history(self, device_id):
        """Get all tests for a specific device"""
        return self.tests_df[self.tests_df['device_id'] == device_id].sort_values('created_dt')
    
    def get_measurements(self, device_id=None, test_name=None):
        """Get measurements with optional filtering"""
        df = self.measurements_df
        
        if device_id:
            df = df[df['device_id'] == device_id]
        if test_name:
            df = df[df['test_name'] == test_name]
        
        return df
    
    def get_series(self, device_id, report_id, test_name):
        """Get time series data for a specific test"""
        key = (device_id, report_id, test_name)
        return self.series_store.get(key, {})

print("TestDataLoader class defined")

TestDataLoader class defined


## 2. Load and Parse Reports

In [12]:
# Initialize data loader and load all reports
loader = TestDataLoader()

num_files = loader.find_reports()
num_reports, num_tests, num_measurements = loader.load_all_reports()

print(f"Found {num_files} report files")
print(f"Loaded {num_reports} reports, {num_tests} tests, {num_measurements} measurements")

# Create short aliases for convenience
reports_df = loader.reports_df
tests_df = loader.tests_df
measurements_df = loader.measurements_df

# Get device summary
device_summary = loader.get_device_summary()
print(f"\nUnique devices: {len(device_summary) if device_summary is not None else 0}")
if device_summary is not None:
    display(device_summary)

Found 22 report files
Loaded 22 reports, 137 tests, 239 measurements

Unique devices: 6


,device_id,num_runs,total_tests,total_passed,total_failed,date_range,pass_rate
0,BYUgAHwA,8,42,39,3,2026-03-23 to 2026-03-24,92.9
1,FC1B13dv,1,8,5,3,2026-03-24 to 2026-03-24,62.5
2,FC1Brsqh,2,14,8,6,2026-03-24 to 2026-03-24,57.1
3,Minion03,2,10,6,4,2026-03-23 to 2026-03-24,60.0
4,PC71ABGT,1,7,7,0,2026-03-24 to 2026-03-24,100.0
5,Pelzm,7,54,26,28,2026-03-23 to 2026-03-24,48.1


## 3. Data Inspection

In [13]:
# Device Summary - Device-centric view
print("=" * 80)
print("DEVICE SUMMARY")
print("=" * 80)

device_summary = loader.get_device_summary()

if device_summary is not None and len(device_summary) > 0:
    print(f"\nTotal unique devices: {len(device_summary)}")
    print(f"Total test runs: {device_summary['num_runs'].sum()}")
    print(f"Overall pass rate: {device_summary['pass_rate'].mean():.1f}%")
    print("\nDevice Details:")
    display(device_summary)
else:
    print("No device data available")

print("\n" + "=" * 80)
print("REPORTS OVERVIEW")
print("=" * 80)
print(f"\nTotal reports: {len(reports_df)}")
print(f"Date range: {reports_df['created_dt'].min()} to {reports_df['created_dt'].max()}")
print(f"\nReports by device:")
display(reports_df.groupby('device_id').agg(
    runs=('report_id', 'count'),
    first_run=('created_dt', 'min'),
    last_run=('created_dt', 'max')
).reset_index())

DEVICE SUMMARY

Total unique devices: 6
Total test runs: 21
Overall pass rate: 70.1%

Device Details:


,device_id,num_runs,total_tests,total_passed,total_failed,date_range,pass_rate
0,BYUgAHwA,8,42,39,3,2026-03-23 to 2026-03-24,92.9
1,FC1B13dv,1,8,5,3,2026-03-24 to 2026-03-24,62.5
2,FC1Brsqh,2,14,8,6,2026-03-24 to 2026-03-24,57.1
3,Minion03,2,10,6,4,2026-03-23 to 2026-03-24,60.0
4,PC71ABGT,1,7,7,0,2026-03-24 to 2026-03-24,100.0
5,Pelzm,7,54,26,28,2026-03-23 to 2026-03-24,48.1



REPORTS OVERVIEW

Total reports: 22
Date range: 2026-03-23 16:56:44.235650 to 2026-03-24 12:12:22.222957

Reports by device:


,device_id,runs,first_run,last_run
0,BYUgAHwA,8,2026-03-23 17:26:37.811580,2026-03-24 12:10:19.951527
1,FC1B13dv,1,2026-03-24 10:51:08.427652,2026-03-24 10:51:08.427652
2,FC1Brsqh,2,2026-03-24 11:02:43.516113,2026-03-24 11:04:45.771376
3,Minion03,2,2026-03-23 16:56:44.235650,2026-03-24 10:18:38.788533
4,PC71ABGT,1,2026-03-24 12:12:22.222957,2026-03-24 12:12:22.222957
5,Pelzm,7,2026-03-23 17:04:59.492941,2026-03-24 11:01:03.378271


In [30]:
# Show sample data
print("\nSample Records:")
display(tests_df.head(10))


Sample Records:


,report_id,device_id,test_name,nodeid,outcome,duration_s,created_dt,failure_analysis_text,Sensor Node Name_text
0,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_power_usage_disconnected,icotest/test/test_sensor_node.py::test_power_u...,passed,1.588535,2026-03-23 17:26:37.811580,"High power draw may indicate a short circuit, ...",NaN
1,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_power_usage_connected,icotest/test/test_sensor_node.py::test_power_u...,passed,1.485093,2026-03-23 17:26:37.811580,High power draw often suggests internal compon...,NaN
2,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_power_usage_streaming,icotest/test/test_sensor_node.py::test_power_u...,passed,1.434958,2026-03-23 17:26:37.811580,High streaming power may indicate RF hardware ...,NaN
3,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_acceleration_sensor_self_test,icotest/test/test_sth.py::test_acceleration_se...,passed,0.554441,2026-03-23 17:26:37.811580,NaN,NaN
4,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_acceleration_single_value,icotest/test/test_sth.py::test_acceleration_si...,passed,0.163586,2026-03-23 17:26:37.811580,NaN,NaN
5,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_acceleration_noise,icotest/test/test_sth.py::test_acceleration_noise,passed,2.608298,2026-03-23 17:26:37.811580,NaN,NaN
6,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_acceleration_3a_alt,icotest/test/test_sth.py::test_acceleration_3a...,passed,3.993032,2026-03-23 17:26:37.811580,NaN,NaN
7,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_acceleration_3a_optimized,icotest/test/test_sth.py::test_acceleration_3a...,passed,2.672764,2026-03-23 17:26:37.811580,NaN,NaN
8,BYUgAHwA_BYUgAHwA_2026-03-23_17-26,BYUgAHwA,test_BaP_torr_accelleration,icotest/test/test_sth.py::test_BaP_torr_accell...,failed,1.302866,2026-03-23 17:26:37.811580,NaN,NaN
9,BYUgAHwA_BYUgAHwA_2026-03-23_19-02,BYUgAHwA,test_power_usage_disconnected,icotest/test/test_sensor_node.py::test_power_u...,passed,1.830257,2026-03-23 19:02:20.018841,"High power draw may indicate a short circuit, ...",NaN


## 4. Interactive Filtering

In [36]:
# Interactive filtering by device
print("=" * 80)
print("FILTER DATA")
print("=" * 80)

# Get unique values
devices = sorted([d for d in reports_df['device_id'].dropna().unique() if pd.notna(d)])
tests = sorted(tests_df['test_name'].unique())
outcomes = sorted(tests_df['outcome'].unique())

# Date range
min_date = reports_df['created_dt'].min()
max_date = reports_df['created_dt'].max()

# Create widgets
device_selector = widgets.SelectMultiple(
    options=devices,
    value=(devices[0],) if devices else (),
    description='Devices:',
    rows=min(5, len(devices))
)

test_selector = widgets.SelectMultiple(
    options=tests,
    description='Tests:',
    rows=min(5, len(tests))
)

outcome_selector = widgets.SelectMultiple(
    options=outcomes,
    value=tuple(outcomes),
    description='Outcomes:',
    rows=min(3, len(outcomes))
)

min_date_widget = widgets.Text(
    value=min_date.strftime('%Y-%m-%d %H:%M') if min_date else '',
    description='From:',
    placeholder='YYYY-MM-DD HH:MM'
)

max_date_widget = widgets.Text(
    value=max_date.strftime('%Y-%m-%d %H:%M') if max_date else '',
    description='To:',
    placeholder='YYYY-MM-DD HH:MM'
)

print("\n1. Select Devices:")
display(device_selector)
print("\n2. Select Tests (leave empty for all):")
display(test_selector)
print("\n3. Select Outcomes:")
display(outcome_selector)
print("\n4. Date Range:")
display(widgets.VBox([min_date_widget, max_date_widget]))

FILTER DATA

1. Select Devices:


SelectMultiple(description='Devices:', index=(0,), options=('BYUgAHwA', 'FC1B13dv', 'FC1Brsqh', 'Minion03', 'P…


2. Select Tests (leave empty for all):


SelectMultiple(description='Tests:', options=('test_BaP_torr_accelleration', 'test_acceleration_3a_alt', 'test…


3. Select Outcomes:


SelectMultiple(description='Outcomes:', index=(0, 1, 2), options=('error', 'failed', 'passed'), rows=3, value=…


4. Date Range:


In [42]:
# Apply filters and get filtered data
def get_filtered_data():
    """Apply all filters and return filtered DataFrame"""
    df = tests_df.copy()
    
    # Filter by device
    if device_selector.value:
        df = df[df['device_id'].isin(device_selector.value)]
    
    # Filter by test
    if test_selector.value:
        df = df[df['test_name'].isin(test_selector.value)]
    
    # Filter by outcome
    if outcome_selector.value:
        df = df[df['outcome'].isin(outcome_selector.value)]
    
    # Filter by date range
    try:
        start_date = datetime.strptime(min_date_widget.value, '%Y-%m-%d %H:%M') if min_date_widget.value else reports_df['created_dt'].min()
        end_date = datetime.strptime(max_date_widget.value, '%Y-%m-%d %H:%M') if max_date_widget.value else reports_df['created_dt'].max()
        
        df = df[(df['created_dt'] >= start_date) & (df['created_dt'] <= end_date)]
    except ValueError:
        print("Invalid date format. Using full range.")
    
    return df.sort_values('created_dt')

df_filtered = get_filtered_data()
print(f"\nFiltered: {len(df_filtered)} test records")
display(df_filtered.head(10))


Filtered: 61 test records


,report_id,device_id,test_name,nodeid,outcome,duration_s,created_dt,failure_analysis_text,Sensor Node Name_text
65,Minion03_BYUgAHwA_2026-03-23_16-56,BYUgAHwA,test_power_usage_connected,icotest/test/test_sensor_node.py::test_power_u...,passed,1.446660,2026-03-23 16:56:44.235650,High power draw often suggests internal compon...,NaN
67,Minion03_BYUgAHwA_2026-03-23_16-56,BYUgAHwA,test_acceleration_sensor_self_test,icotest/test/test_sth.py::test_acceleration_se...,passed,0.560697,2026-03-23 16:56:44.235650,NaN,NaN
69,Minion03_BYUgAHwA_2026-03-23_16-56,BYUgAHwA,test_acceleration_noise,icotest/test/test_sth.py::test_acceleration_noise,passed,2.571201,2026-03-23 16:56:44.235650,NaN,NaN
64,Minion03_BYUgAHwA_2026-03-23_16-56,BYUgAHwA,test_power_usage_disconnected,icotest/test/test_sensor_node.py::test_power_u...,passed,2.978781,2026-03-23 16:56:44.235650,"High power draw may indicate a short circuit, ...",NaN
81,Pelzm_04_BYUgAHwA_2026-03-23_17-04,BYUgAHwA,test_power_usage_disconnected,icotest/test/test_sensor_node.py::test_power_u...,passed,1.564184,2026-03-23 17:04:59.492941,"High power draw may indicate a short circuit, ...",NaN
82,Pelzm_04_BYUgAHwA_2026-03-23_17-04,BYUgAHwA,test_power_usage_connected,icotest/test/test_sensor_node.py::test_power_u...,passed,1.483344,2026-03-23 17:04:59.492941,High power draw often suggests internal compon...,NaN
84,Pelzm_04_BYUgAHwA_2026-03-23_17-04,BYUgAHwA,test_acceleration_sensor_self_test,icotest/test/test_sth.py::test_acceleration_se...,passed,0.471054,2026-03-23 17:04:59.492941,NaN,NaN
86,Pelzm_04_BYUgAHwA_2026-03-23_17-04,BYUgAHwA,test_acceleration_noise,icotest/test/test_sth.py::test_acceleration_noise,passed,2.613366,2026-03-23 17:04:59.492941,NaN,NaN
93,Pelzm_04_BYUgAHwA_2026-03-23_17-07,BYUgAHwA,test_acceleration_sensor_self_test,icotest/test/test_sth.py::test_acceleration_se...,passed,0.487148,2026-03-23 17:07:14.578217,NaN,NaN
91,Pelzm_04_BYUgAHwA_2026-03-23_17-07,BYUgAHwA,test_power_usage_connected,icotest/test/test_sensor_node.py::test_power_u...,passed,1.478696,2026-03-23 17:07:14.578217,High power draw often suggests internal compon...,NaN


## 5. Device Drill-Down

Select a device to see its complete test history and measurements.

In [41]:
# Device Drill-Down
print("=" * 80)
print("DEVICE DRILL-DOWN")
print("=" * 80)

# Create device selector
device_dropdown = widgets.Dropdown(
    options=[d for d in devices if pd.notna(d)],
    description='Device:',
    style={'description_width': '80px'}
)

display(device_dropdown)

def show_device_details(device_id):
    """Show all tests and measurements for a selected device"""
    if device_id is None:
        print("Select a device")
        return
    
    # Get device data
    device_tests = tests_df[tests_df['device_id'] == device_id].sort_values('created_dt')
    device_reports = reports_df[reports_df['device_id'] == device_id]
    device_measurements = measurements_df[measurements_df['device_id'] == device_id]
    
    print(f"
{'='*60}")
    print(f"Device: {device_id}")
    print(f"{'='*60}")
    
    # Summary stats
    total = len(device_tests)
    passed = (device_tests['outcome'] == 'passed').sum()
    failed = (device_tests['outcome'] == 'failed').sum()
    errors = (device_tests['outcome'] == 'error').sum()
    pass_rate = (passed / total * 100) if total > 0 else 0
    
    print(f"
Summary:")
    print(f"  Reports: {len(device_reports)}")
    print(f"  Tests: {total}")
    print(f"  Passed: {passed}, Failed: {failed}, Errors: {errors}")
    print(f"  Pass Rate: {pass_rate:.1f}%")
    
    # Test history table
    print(f"
Test History:")
    display(device_tests[['created_dt', 'test_name', 'outcome', 'duration_s']].head(20))
    
    # Recent measurements
    if len(device_measurements) > 0:
        print(f"
Recent Measurements:")
        recent = device_measurements.dropna(subset=['value']).sort_values('report_id').groupby('metric').tail(1)
        display(recent[['metric', 'value', 'unit']].head(10))
    
    # Optional: plot pass/fail over time
    fig, ax = plt.subplots(figsize=(10, 4))
    outcome_counts = device_tests.groupby(['created_dt', 'outcome']).size().unstack(fill_value=0)
    if len(outcome_counts.columns) > 0:
        outcome_counts.plot(kind='bar', stacked=True, ax=ax, alpha=0.7)
        ax.set_title(f'Test Outcomes Over Time - {device_id}')
        ax.set_xlabel('Date')
        ax.set_ylabel('Count')
        ax.legend(title='Outcome')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

# Create interactive output and display it
out = widgets.interactive_output(show_device_details, {'device_id': device_dropdown})
display(out)

METRIC ANALYSIS

Available metrics: ['power_usage_disconnected', 'power_usage_connected', 'power_usage_streaming', 'self_test_voltage_abs', 'self_test_voltage_drift', 'acceleration_noise_snr', 'acceleration_mean', 'acceleration_x_bias', 'acceleration_x_snr', 'acceleration_y_bias', 'acceleration_y_snr', 'acceleration_z_bias', 'acceleration_z_snr', 'acceleration_vector_error', 'acceleration_noise_margin', 'backpack_acc_x_bias', 'backpack_acc_y_bias', 'backpack_acc_torr_bias', 'backpack_acc_x_snr', 'backpack_acc_y_snr', 'backpack_acc_torr_snr', 'backpack_max_bias', 'backpack_max_noise']


Dropdown(description='Metric:', options=('acceleration_mean', 'acceleration_noise_margin', 'acceleration_noise…

## 6. Device Comparison

In [ ]:
# Metric Analysis - View measurements for selected devices
print("=" * 80)
print("METRIC ANALYSIS")
print("=" * 80)

# Get metrics from measurements
if len(measurements_df) > 0:
    numeric_metrics = sorted(measurements_df[measurements_df['value'].notna()]['metric'].unique())
    print(f"Available metrics: {numeric_metrics}")
    
    metric_dropdown = widgets.Dropdown(
        options=numeric_metrics,
        description='Metric:',
        style={'description_width': '80px'}
    )
    display(metric_dropdown)
    
    def plot_metric(metric):
        selected = device_selector.value if device_selector.value else tuple([d for d in devices if pd.notna(d)])
        data = measurements_df[(measurements_df['metric'] == metric) & (measurements_df['device_id'].isin(selected))].dropna(subset=['value'])
        
        if len(data) == 0:
            print(f"No data for {metric}")
            return
        
        # Plot by device over time
        fig, ax = plt.subplots(figsize=(12, 5))
        
        for dev in data['device_id'].unique():
            dev_data = data[data['device_id'] == dev].sort_values('report_id')
            ax.plot(range(len(dev_data)), dev_data['value'].values, marker='o', label=dev, alpha=0.7)
        
        ax.set_xlabel('Test Run')
        ax.set_ylabel(metric)
        ax.set_title(f'{metric} Over Time')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Stats
        print(f"
{metric} Statistics:")
        display(data.groupby('device_id')['value'].describe())
    
    out = widgets.interactive_output(plot_metric, {'metric': metric_dropdown})
    display(out)
else:
    print("No measurements available")

In [22]:
# Device Comparison
print("=" * 80)
print("DEVICE COMPARISON")
print("=" * 80)

# Compare metrics across devices
if len(measurements_df) > 0:
    numeric_metrics = measurements_df[measurements_df['value'].notna()]['metric'].unique()
    
    if len(numeric_metrics) > 0:
        comp_metric = widgets.Dropdown(
            options=sorted(numeric_metrics),
            description='Compare:'
        )
        display(comp_metric)
        
        def compare_devices(metric):
            data = measurements_df[measurements_df['metric'] == metric].dropna(subset=['value'])
            
            if len(data) == 0:
                print(f"No data for {metric}")
                return
            
            # Box plot by device
            devices_with_data = data.groupby('device_id').filter(lambda x: len(x) > 1)['device_id'].unique()
            
            if len(devices_with_data) < 2:
                print("Need multiple devices with data to compare")
                return
            
            fig, ax = plt.subplots(figsize=(10, 5))
            data_to_plot = data[data['device_id'].isin(devices_with_data)]
            data_to_plot.boxplot(column='value', by='device_id', ax=ax)
            ax.set_title(f'{metric} Comparison by Device')
            ax.set_xlabel('Device')
            ax.set_ylabel(metric)
            plt.suptitle('')
            plt.tight_layout()
            plt.show()
            
            # Show summary stats
            print(f"\n{metric} by Device:")
            display(data.groupby('device_id')['value'].describe())
        
        out = widgets.interactive_output(compare_devices, {'metric': comp_metric})
display(out)
    else:
        print("No numeric metrics available for comparison")
else:
    print("No measurements loaded")

DEVICE COMPARISON


Dropdown(description='Compare:', options=('acceleration_mean', 'acceleration_noise_margin', 'acceleration_nois…

## 7. Time Series Deep Dive

In [23]:
# Time Series Browser
print("=" * 80)
print("TIME SERIES DATA")
print("=" * 80)

# Find tests with time series data
series_keys = list(loader.series_store.keys())

if series_keys:
    print(f"Found {len(series_keys)} tests with time series data")
    
    # Create selector options
    options = [(f"{d} | {r} | {t}", (d, r, t)) for d, r, t in series_keys]
    
    series_selector = widgets.Dropdown(
        options=options,
        description='Select:'
    )
    display(series_selector)
    
    def show_series(selection):
        device_id, report_id, test_name = selection
        series = loader.get_series(device_id, report_id, test_name)
        
        if not series:
            print("No time series data")
            return
        
        print(f"\nDevice: {device_id}")
        print(f"Report: {report_id}")
        print(f"Test: {test_name}")
        print(f"\nAvailable series: {list(series.keys())}")
        
        # Plot each series
        for name, data in series.items():
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(data, alpha=0.7)
            ax.set_title(f'{name}')
            ax.set_xlabel('Sample')
            ax.set_ylabel('Value')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            print(f"  {name}: len={len(data)}, mean={np.mean(data):.2f}, std={np.std(data):.2f}")
    
    out = widgets.interactive_output(show_series, {'selection': series_selector})
display(out)
else:
    print("No time series data available")

TIME SERIES DATA
Found 31 tests with time series data


Dropdown(description='Select:', options=(('BYUgAHwA | BYUgAHwA_BYUgAHwA_2026-03-23_17-26 | test_acceleration_n…

In [25]:
# Time series already displayed above in the browser

NameError: name 'tests_with_series' is not defined

## 8. Raw DataFrame Access

In [26]:
# Access filtered data for custom analysis

print("Use the following variables for custom analysis:")
print("\n  tests_df         - Full DataFrame with all records")
print("  get_filtered_data() - Function to get filtered DataFrame")
print("  loader.get_series(filename, timestamp, test_name) - Get time series data")

print("\n" + "="*80)
print("Current Filtered Data:")
print("="*80)

df_current = tests_df.copy()
print(f"\nShape: {df_current.shape}")
print(f"Columns: {list(df_current.columns)}")

# Show summary statistics
print("\nSummary Statistics:")
display(df_current.describe())

Use the following variables for custom analysis:

  tests_df         - Full DataFrame with all records
  get_filtered_data() - Function to get filtered DataFrame
  loader.get_series(filename, timestamp, test_name) - Get time series data

Current Filtered Data:


NameError: name 'get_filtered_data' is not defined

In [ ]:
# Display full filtered data
print("Full Filtered Data:")
display(df_current)

## 9. Helper Functions for Custom Analysis

In [ ]:
# Helper Functions for Custom Analysis
print("=" * 80)
print("HELPER FUNCTIONS")
print("=" * 80)

def get_device(device_id):
    """Get all test data for a device"""
    return tests_df[tests_df['device_id'] == device_id].sort_values('created_dt')

def get_report(report_id):
    """Get all test data for a report"""
    return tests_df[tests_df['report_id'] == report_id]

def get_test(test_name, device_id=None):
    """Get all runs of a specific test"""
    df = tests_df[tests_df['test_name'] == test_name]
    if device_id:
        df = df[df['device_id'] == device_id]
    return df.sort_values('created_dt')

def get_measurement(metric, device_id=None):
    """Get all measurements for a specific metric"""
    df = measurements_df[measurements_df['metric'] == metric]
    if device_id:
        df = df[df['device_id'] == device_id]
    return df.dropna(subset=['value'])

def device_stats(device_id):
    """Get statistics for a device"""
    device_tests = tests_df[tests_df['device_id'] == device_id]
    
    stats = {
        'Total Tests': len(device_tests),
        'Passed': (device_tests['outcome'] == 'passed').sum(),
        'Failed': (device_tests['outcome'] == 'failed').sum(),
        'Errors': (device_tests['outcome'] == 'error').sum(),
        'Pass Rate': f"{(device_tests['outcome'] == 'passed').sum() / len(device_tests) * 100:.1f}%"
    }
    
    return pd.Series(stats)

print("Available helper functions:")
print("  - get_device(device_id)          # All tests for a device")
print("  - get_report(report_id)          # All tests in a report")
print("  - get_test(test_name, device_id) # All runs of a test")
print("  - get_measurement(metric, device_id) # All measurements")
print("  - device_stats(device_id)        # Device statistics")

# Example usage
if devices:
    example_device = devices[0]
    print(f"\nExample: Statistics for {example_device}")
    display(device_stats(example_device))

In [ ]:
# Example: Get statistics for a specific test

# Choose a test
example_test = tests_df['test_name'].value_counts().index[0]
print(f"Example: Statistics for '{example_test}'\n")

stats = get_test_statistics(example_test)
display(stats)